# Jais Voice — الوكيل الصوتي على Kaggle 🎙️

يربط 3 نماذج في خط أنبابة واحد: **ASR → LLM → TTS** ثم ينشر الخادم عبر رابط عام (CloudFlare Quick Tunnel).

**قبل التشغيل:**
1. Settings (الشريط الجانبي) → Accelerator = **GPU T4 ×2**
2. Settings → Internet = **ON**
3. Run All

في النهاية يظهر لك رابط `https://xxxx.trycloudflare.com` — افتحه من أي جهاز وتحدث مع الوكيل بالصوت.

> ⏱ أول رد قد يتأخذ من دقيقة إلى دقيقتين (تحميل النموذج). مكالمات لاحقة أسرع بكثير.

In [ ]:
CONFIG = {
    "PORT": 8000,
    "ASR_SIZE": "small",          # faster-whisper: tiny/base/small/medium
    "ASR_DEVICE": "cpu",          # T4 busy by LLM — asr on cpu is fast
    "LLM_MODEL": "inception42/Jais-3-7B",   # أو مسار محلي بعد snapshot_download
    "TTS_VOICE": "ar-SA-HamedNeural",        # بديل: ar-EG-SalmaNeural | ar-EG-ShakirNeural
    "HF_TOKEN": "",               # اختياري للنماذج المُقيّدة
    "NGROK_TOKEN": "",            # احتياطي للنفق إن فشل cloudflared
}
import json, os
os.makedirs("/kaggle/working", exist_ok=True)
open("/kaggle/working/.jais_config.json", "w").write(json.dumps(CONFIG, ensure_ascii=False))
if CONFIG["HF_TOKEN"]:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = CONFIG["HF_TOKEN"]
print("✅ CONFIG تم")

In [ ]:
# ==== 1) تثبيت الاعتماديات ====
!pip -q install fastapi uvicorn faster-whisper edge-tts pydantic-settings soundfile av accelerate 2>&1 | tail -n 2
print("✅ Deps installed")

In [ ]:
# ==== 2) جلب كود المشروع من GitHub ====
import subprocess, os
REPO = "/kaggle/working/voice-agent"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "https://github.com/911houssem/jais-voice-agent.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=False)
print("✅ Repo ready")

### (اختياري) تحميل النماذج إلى القرص بدل Ver من HuggingFace مباشرة

الخلية التالية تنزّل Jais-3-7B لمسار محلي؛ ثم غيّر `LLM_MODEL` في CONFIG لمسار المجلد المحلي.

In [ ]:
# OPTIONAL: snapshot_download("inception42/Jais-3-7B", local_dir="/kaggle/working/jais3-7b", token=CONFIG["HF_TOKEN"])
# if os.path.exists("/kaggle/working/jais3-7b"):
#     CONFIG["LLM_MODEL"] = "/kaggle/working/jais3-7b"
#     open("/kaggle/working/.jais_config.json","w").write(json.dumps(CONFIG, ensure_ascii=False))
print("skip (اختياري)")

In [ ]:
# ==== 3) بدء الخادم (uvicorn في خيط منفصل) ====
import sys, time
sys.path.insert(0, "/kaggle/working/voice-agent/kaggle")
import kaggle_app
kaggle_app.start_app(CONFIG["PORT"])
time.sleep(2)
import urllib.request
print(urllib.request.urlopen(f"http://127.0.0.1:{CONFIG['PORT']}/health").read().decode())

In [ ]:
# ==== 4) نفق عام (Cloudflare Quick Tunnel) ====
import subprocess, re, os, threading, time

def start_cloudflared(port):
    binpath = "/kaggle/working/cloudflared"
    if not os.path.exists(binpath):
        subprocess.run(["bash", "-c",
            f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {binpath} && chmod +x {binpath}"], check=True)
    args = [binpath, "tunnel", "--url", f"http://localhost:{port}", "--no-autoupdate"]
    if use_http2:
        args += ["--protocol", "http2"]
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url = None
    def _drain():
        nonlocal url
        for line in proc.stdout:
            if url is None:
                m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
                if m: url = m.group(0)
    th = threading.Thread(target=_drain, daemon=True); th.start()
    for _ in range(180):
        if url: break
        time.sleep(1)
    return proc, url

proc, url = start_cloudflared(CONFIG["PORT"])
if url:
    from IPython.display import display, HTML
    display(HTML(f'<h3>🔗 رابطك العام: <a href="{url}" target="_blank">{url}</a></h3><p>افتح الرابط من أي جهاز واضغط <b>اتصال</b> ثم <b>تحدث</b> واسأل الوكيل.</p>'))
else:
    print("فشل cloudflared — استخدم خلية ngrok الاحتياطية تحت")

In [ ]:
# ==== (احتياطي) نفق ngrok إذا فشل cloudflared ====
if not url and CONFIG.get("NGROK_TOKEN"):
    !pip -q install pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(CONFIG["NGROK_TOKEN"])
    pub = ngrok.connect(CONFIG["PORT"], "http").public_url
    print("🔗", pub)
else:
    print("skip 屏幕ngrok (تحقق من الخلية السابقة أو فعل NGROK_TOKEN في CONFIG)")

In [ ]:
# ==== 5) الحفاظ على الجلسة حيّة (اتركها تعمل) ====
import time, datetime
while True:
    print("[alive]", datetime.datetime.now().strftime("%H:%M:%S"), flush=True)
    time.sleep(120)